# Standard deviation estimates for Auswertung 1608

This notebook computes standard deviation estimates for the aggregate metrics reported in `experimente.tex` and `experimentepins.tex` from the per-sample CSV files in `eva_server/Auswertung_1608`.

Formula used here:

- `SD`: sample standard deviation across per-sample rows (`ddof=1`)
- The table reports only `mean ± SD`.


The notebook does not modify the TeX files. It writes separate uncertainty tables to `eva_server/Auswertung_1608/uncertainty_outputs/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE = Path("eva_server/Auswertung_1608")
OUT = BASE / "uncertainty_outputs"
OUT.mkdir(parents=True, exist_ok=True)

METRICS = {
    "dice_mean": "Dice",
    "rmse_mean": "RMSE",
    "rmse_whole_mean": "RMSE whole",
    "soft_intensity_dice_whole_mean": "Soft Dice whole",
    "spot_recognition_mean": "Recognition",
}

MODEL_LABELS = {
    # Two-output table
    "final_20260719-084743_augmented_spots_train_e200_b100_lr0.0001_s1": "Two-output L1 loss, batch size 100",
    "final_20260721-011126_augmented_spots_train_e200_b10_lr0.0001_s1": "Two-output L1 loss",
    "final_20260717-084426_augmented_spots_train_e200_b10_lr0.0001_s1": "Two-output L1 loss, 32 base channels",
    "final_20260721-102717_to_newloss_augmented_spots_train_e200_b10_lr0.0001_s1": "Two-output reconstruction loss",
    "final_20260707-111849_twoouts_augmented_spots_train_e200_b10_lr0.0001_s1": "Two-output Tversky loss",
    "final_20260703-201335_augmented_spot_patches_e200_b10_lr0.0001_s1": "Two-output Tversky loss, small dataset",
    # One-output table
    "final_20260702-145153_augmented_spots_train_e200_b10_lr0.0001_s1": "One-output Dice/L1 loss",
    "final_20260727-152943_oneoutput_augmented_spots_train_e2000_b10_lr0.0001_s1": "One-output Tversky loss",
    # Multi-output table
    "final_20260804-224906_augmented_spots_train_e200_b20_lr0.0002_s1": "Multi-output Tversky loss, ReduceLROnPlateau factor 0.5",
    "final_20260803-135417_augmented_spots_train_e200_b10_lr0.0001_s1": "Multi-output Tversky loss, learning rate 1e-4",
    "final_20260802-123726_augmented_spots_train_e200_b20_lr0.0001_s1": "Multi-output Tversky loss, batch size 20",
    "final_20260729-043315_augmented_spots_train_e200_b10_lr5e-05_s1": "Multi-output Tversky loss, learning rate 5e-5",
    "final_20260721-204034_augmented_spots_train_e200_b100_lr0.0001_s1": "Multi-output Tversky loss, batch size 100",
    "final_20260720-153845_augmented_spots_train_e200_b10_lr0.0001_s1": "Multi-output Tversky loss, 32 base channels",
    "final_20260728-073240_augmented_spots_train_e200_b10_lr0.0003_s1": "Multi-output Tversky loss, learning rate 3e-4",
    "final_20260709-141342_augmented_spots_train_pin_e200_b10_lr0.0001_s1": "Multi-output BCE/MSE loss",
    "final_20260704-022731_augmented_spot_patches_with_masks_e200_b10_lr0.0001_s1": "Multi-output BCE/MSE loss, small dataset",
}

PINNED_TABLE_VALUES = {
    ("Two-output L1 loss, batch size 100", "Dice"): 0.9222,
    ("Two-output L1 loss, batch size 100", "RMSE"): 0.0482,
    ("Two-output L1 loss, batch size 100", "RMSE whole"): 0.0096,
    ("Two-output L1 loss, batch size 100", "Soft Dice whole"): 0.9419,
    ("Two-output L1 loss, batch size 100", "Recognition"): 0.9370,
    ("Two-output L1 loss", "Dice"): 0.9156,
    ("Two-output L1 loss", "RMSE"): 0.0510,
    ("Two-output L1 loss", "RMSE whole"): 0.0109,
    ("Two-output L1 loss", "Soft Dice whole"): 0.9327,
    ("Two-output L1 loss", "Recognition"): 0.9323,
    ("Two-output L1 loss, 32 base channels", "Dice"): 0.8784,
    ("Two-output L1 loss, 32 base channels", "RMSE"): 0.0648,
    ("Two-output L1 loss, 32 base channels", "RMSE whole"): 0.0146,
    ("Two-output L1 loss, 32 base channels", "Soft Dice whole"): 0.9009,
    ("Two-output L1 loss, 32 base channels", "Recognition"): 0.9068,
    ("Two-output reconstruction loss", "Dice"): 0.8800,
    ("Two-output reconstruction loss", "RMSE"): 0.0407,
    ("Two-output reconstruction loss", "RMSE whole"): 0.0099,
    ("Two-output reconstruction loss", "Soft Dice whole"): 0.9515,
    ("Two-output reconstruction loss", "Recognition"): 0.9197,
    ("Two-output Tversky loss", "Dice"): 0.9627,
    ("Two-output Tversky loss", "RMSE"): 0.6664,
    ("Two-output Tversky loss", "RMSE whole"): 0.1544,
    ("Two-output Tversky loss", "Soft Dice whole"): 0.5341,
    ("Two-output Tversky loss", "Recognition"): 0.6481,
    ("Two-output Tversky loss, small dataset", "Dice"): 0.7293,
    ("Two-output Tversky loss, small dataset", "RMSE"): 0.5783,
    ("Two-output Tversky loss, small dataset", "RMSE whole"): 0.1553,
    ("Two-output Tversky loss, small dataset", "Soft Dice whole"): 0.4926,
    ("Two-output Tversky loss, small dataset", "Recognition"): 0.5755,
    ("One-output Dice/L1 loss", "Dice"): 0.6843,
    ("One-output Dice/L1 loss", "RMSE"): 0.1350,
    ("One-output Dice/L1 loss", "RMSE whole"): 0.0337,
    ("One-output Dice/L1 loss", "Soft Dice whole"): 0.7997,
    ("One-output Dice/L1 loss", "Recognition"): 0.7747,
    ("One-output Tversky loss", "Dice"): 0.6291,
    ("One-output Tversky loss", "RMSE"): 0.1652,
    ("One-output Tversky loss", "RMSE whole"): 0.0390,
    ("One-output Tversky loss", "Soft Dice whole"): 0.6314,
    ("One-output Tversky loss", "Recognition"): 0.7319,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5", "Dice"): 0.9411,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5", "RMSE"): 0.0396,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5", "RMSE whole"): 0.0084,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5", "Soft Dice whole"): 0.9758,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5", "Recognition"): 0.9507,
    ("Multi-output Tversky loss, learning rate 1e-4", "Dice"): 0.9411,
    ("Multi-output Tversky loss, learning rate 1e-4", "RMSE"): 0.0432,
    ("Multi-output Tversky loss, learning rate 1e-4", "RMSE whole"): 0.0104,
    ("Multi-output Tversky loss, learning rate 1e-4", "Soft Dice whole"): 0.9728,
    ("Multi-output Tversky loss, learning rate 1e-4", "Recognition"): 0.9490,
    ("Multi-output Tversky loss, batch size 20", "Dice"): 0.9380,
    ("Multi-output Tversky loss, batch size 20", "RMSE"): 0.0401,
    ("Multi-output Tversky loss, batch size 20", "RMSE whole"): 0.0096,
    ("Multi-output Tversky loss, batch size 20", "Soft Dice whole"): 0.9729,
    ("Multi-output Tversky loss, batch size 20", "Recognition"): 0.9490,
    ("Multi-output Tversky loss, learning rate 5e-5", "Dice"): 0.9349,
    ("Multi-output Tversky loss, learning rate 5e-5", "RMSE"): 0.0409,
    ("Multi-output Tversky loss, learning rate 5e-5", "RMSE whole"): 0.0094,
    ("Multi-output Tversky loss, learning rate 5e-5", "Soft Dice whole"): 0.9686,
    ("Multi-output Tversky loss, learning rate 5e-5", "Recognition"): 0.9470,
    ("Multi-output Tversky loss, batch size 100", "Dice"): 0.9332,
    ("Multi-output Tversky loss, batch size 100", "RMSE"): 0.0411,
    ("Multi-output Tversky loss, batch size 100", "RMSE whole"): 0.0095,
    ("Multi-output Tversky loss, batch size 100", "Soft Dice whole"): 0.9704,
    ("Multi-output Tversky loss, batch size 100", "Recognition"): 0.9460,
    ("Multi-output Tversky loss, 32 base channels", "Dice"): 0.9294,
    ("Multi-output Tversky loss, 32 base channels", "RMSE"): 0.0431,
    ("Multi-output Tversky loss, 32 base channels", "RMSE whole"): 0.0104,
    ("Multi-output Tversky loss, 32 base channels", "Soft Dice whole"): 0.9685,
    ("Multi-output Tversky loss, 32 base channels", "Recognition"): 0.9432,
    ("Multi-output Tversky loss, learning rate 3e-4", "Dice"): 0.9086,
    ("Multi-output Tversky loss, learning rate 3e-4", "RMSE"): 0.0492,
    ("Multi-output Tversky loss, learning rate 3e-4", "RMSE whole"): 0.0119,
    ("Multi-output Tversky loss, learning rate 3e-4", "Soft Dice whole"): 0.9700,
    ("Multi-output Tversky loss, learning rate 3e-4", "Recognition"): 0.9297,
    ("Multi-output BCE/MSE loss", "Dice"): 0.8214,
    ("Multi-output BCE/MSE loss", "RMSE"): 0.1273,
    ("Multi-output BCE/MSE loss", "RMSE whole"): 0.0309,
    ("Multi-output BCE/MSE loss", "Soft Dice whole"): 0.7869,
    ("Multi-output BCE/MSE loss", "Recognition"): 0.8471,
    ("Multi-output BCE/MSE loss, small dataset", "Dice"): 0.6439,
    ("Multi-output BCE/MSE loss, small dataset", "RMSE"): 0.1460,
    ("Multi-output BCE/MSE loss, small dataset", "RMSE whole"): 0.0497,
    ("Multi-output BCE/MSE loss, small dataset", "Soft Dice whole"): 0.6283,
    ("Multi-output BCE/MSE loss, small dataset", "Recognition"): 0.7490,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5 evaluation", "Dice"): 0.9300,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5 evaluation", "RMSE"): 0.0460,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5 evaluation", "RMSE whole"): 0.0105,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5 evaluation", "Soft Dice whole"): 0.9581,
    ("Multi-output Tversky loss, ReduceLROnPlateau factor 0.5 evaluation", "Recognition"): 0.9398,
}

SOURCES = [
    ("two_output", BASE / "twooutput" / "model_separation_metrics_5000.csv", "validation_or_train_split_in_file"),
    ("one_output", BASE / "oneoutput" / "oneoutput_model_separation_metrics_5000.csv", "validation_or_train_split_in_file"),
    ("multi_output", BASE / "multi-output" / "pin_model_separation_metrics_5000.csv", "validation_or_train_split_in_file"),
    ("final_test", BASE / "best_model" / "pin_model_separation_metrics_5000.csv", "held_out_test"),
]

In [ ]:
def uncertainty_for_group(group: pd.DataFrame, metric_col: str) -> dict:
    values = group[metric_col].dropna().astype(float)
    n = int(values.shape[0])
    mean = values.mean()
    sd = values.std(ddof=1)
    return {
        "n": n,
        "mean": mean,
        "sd": sd,
    }

rows = []
for source_name, path, evaluation_set in SOURCES:
    df = pd.read_csv(path)
    df = df[df["model"].isin(MODEL_LABELS)].copy()
    for model_id, group in df.groupby("model", sort=False):
        label = MODEL_LABELS[model_id]
        if source_name == "final_test":
            label = "Multi-output Tversky loss, ReduceLROnPlateau factor 0.5 evaluation"
        for metric_col, metric_label in METRICS.items():
            stats = uncertainty_for_group(group, metric_col)
            pinned_value = PINNED_TABLE_VALUES.get((label, metric_label), np.nan)
            rows.append({
                "source": source_name,
                "evaluation_set": evaluation_set,
                "model_label": label,
                "model_id": model_id,
                "metric": metric_label,
                "metric_column": metric_col,
                **stats,
                "pinned_table_value": pinned_value,
                "mean_minus_pinned": stats["mean"] - pinned_value if pd.notna(pinned_value) else np.nan,
                "matches_pinned_to_4dp": bool(pd.notna(pinned_value) and round(stats["mean"], 4) == round(pinned_value, 4)),
            })

uncertainty = pd.DataFrame(rows)
uncertainty.head()

In [ ]:
def fmt_mean_sd(row: pd.Series, decimals: int = 4) -> str:
    return f"{row['mean']:.{decimals}f} +/- {row['sd']:.{decimals}f}"

uncertainty["mean_sd"] = uncertainty.apply(fmt_mean_sd, axis=1)
uncertainty["sd_fmt"] = uncertainty["sd"].map(lambda x: f"{x:.4f}")

wide_sd_table = (
    uncertainty
    .pivot_table(index=["source", "evaluation_set", "model_label", "model_id"], columns="metric", values="mean_sd", aggfunc="first")
    .reset_index()
)

wide_sd = (
    uncertainty
    .pivot_table(index=["source", "evaluation_set", "model_label", "model_id"], columns="metric", values="sd_fmt", aggfunc="first")
    .reset_index()
)

metric_order = ["Dice", "RMSE", "RMSE whole", "Soft Dice whole", "Recognition"]
base_cols = ["source", "evaluation_set", "model_label", "model_id"]
wide_sd_table = wide_sd_table[base_cols + metric_order]
wide_sd = wide_sd[base_cols + metric_order]

uncertainty.to_csv(OUT / "uncertainty_long.csv", index=False)
wide_sd_table.to_csv(OUT / "standard_deviation_wide.csv", index=False)
wide_sd.to_csv(OUT / "uncertainty_sd_wide.csv", index=False)

wide_sd_table

In [ ]:
mismatches = uncertainty.loc[
    uncertainty["pinned_table_value"].notna() & ~uncertainty["matches_pinned_to_4dp"],
    ["source", "model_label", "metric", "mean", "pinned_table_value", "mean_minus_pinned", "n"],
].copy()

mismatches.to_csv(OUT / "pinned_value_mismatches.csv", index=False)
mismatches

In [ ]:
def latex_escape(value: str) -> str:
    return str(value).replace("_", r"\_")

latex_rows = wide_sd_table.copy()
latex_rows["model_label"] = latex_rows["model_label"].map(latex_escape)
latex_rows = latex_rows.drop(columns=["source", "model_id", "evaluation_set"])
latex_rows = latex_rows.rename(columns={"model_label": "Model"})

columns = list(latex_rows.columns)
row_end = " " + chr(92) * 2
lines = [
    r"\begin{tabular}{lccccc}",
    r"\toprule",
    " & ".join(columns) + row_end,
    r"\midrule",
]
for _, row in latex_rows.iterrows():
    lines.append(" & ".join(str(row[col]) for col in columns) + row_end)
lines.extend([r"\bottomrule", r"\end{tabular}"])
latex = "\n".join(lines) + "\n"
(OUT / "standard_deviation_table.tex").write_text(latex)

print(f"Wrote outputs to: {OUT.resolve()}")

## Thesis wording for standard deviations

The uncertainty reported in the evaluation tables is the sample standard deviation across the evaluated samples. For each model and metric, the metric was first computed for every sample in the evaluation set. The table reports the arithmetic mean together with the sample standard deviation, using n - 1 degrees of freedom. Thus, the value after +/- describes the spread of per-sample performance around the mean and not the uncertainty of the mean estimate.

## Note on the final test Dice value

The pinned TeX table reports Dice `0.9300` for the held-out final-model test evaluation. The available per-sample files in `eva_server/Auswertung_1608/best_model` give Dice about `0.9255` from both `pin_model_separation_metrics_5000.csv` and `pin_spot_separation_metrics_5000.csv`. The mismatch is written to `pinned_value_mismatches.csv` so it can be checked before the thesis table is updated.